# 🏆 01 — Exploratory Data Analysis
## FIFA World Cup 2026 Prediction Project

Notebook ini mengeksplorasi semua dataset yang tersedia untuk memahami:
- Distribusi dan kualitas data
- Tren historis pertandingan internasional
- Pola kemenangan di World Cup
- Home advantage effect
- Profil tim peserta WC 2026

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = Path('../data')
print('📂 Data files:', [f.name for f in DATA_DIR.glob('*.csv')])

📂 Data files: ['2022_world_cup_groups.csv', '2022_world_cup_matches.csv', '2022_world_cup_squads.csv', '2026_world_cup_groups.csv', 'data_dictionary.csv', 'international_matches.csv', 'world_cups.csv', 'world_cup_matches.csv']


## 1. Load Data

In [ ]:
# Load semua dataset
intl_matches = pd.read_csv(DATA_DIR / 'international_matches.csv')
wc_matches = pd.read_csv(DATA_DIR / 'world_cup_matches.csv')
world_cups = pd.read_csv(DATA_DIR / 'world_cups.csv')
wc2022_groups = pd.read_csv(DATA_DIR / '2022_world_cup_groups.csv')
wc2022_matches = pd.read_csv(DATA_DIR / '2022_world_cup_matches.csv')
wc2022_squads = pd.read_csv(DATA_DIR / '2022_world_cup_squads.csv', encoding='latin-1')
wc2026_groups = pd.read_csv(DATA_DIR / '2026_world_cup_groups.csv')

print(f'International Matches: {intl_matches.shape}')
print(f'World Cup Matches:     {wc_matches.shape}')
print(f'World Cups Summary:    {world_cups.shape}')
print(f'WC 2022 Groups:        {wc2022_groups.shape}')
print(f'WC 2022 Squads:        {wc2022_squads.shape}')
print(f'WC 2026 Groups:        {wc2026_groups.shape}')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe1 in position 86: invalid continuation byte

## 2. Data Quality Check

In [2]:
print('=== International Matches ===')
print(intl_matches.info())
print('\nMissing values:')
print(intl_matches.isnull().sum())
print('\nFirst 5 rows:')
intl_matches.head()

=== International Matches ===


NameError: name 'intl_matches' is not defined

In [ ]:
# Parse dates
intl_matches['Date'] = pd.to_datetime(intl_matches['Date'])
wc_matches['Date'] = pd.to_datetime(wc_matches['Date'])

print(f'International matches date range: {intl_matches["Date"].min()} to {intl_matches["Date"].max()}')
print(f'World Cup matches date range:     {wc_matches["Date"].min()} to {wc_matches["Date"].max()}')
print(f'\nTournament types: {intl_matches["Tournament"].nunique()} unique tournaments')
print(f'Unique teams:      {pd.concat([intl_matches["Home Team"], intl_matches["Away Team"]]).nunique()}')

In [ ]:
# Top 20 Turnamen
print('Top 20 Tournaments by match count:')
intl_matches['Tournament'].value_counts().head(20)

## 3. Distribusi Gol

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Home Goals distribution
axes[0].hist(intl_matches['Home Goals'].dropna(), bins=range(0, 15), 
             color='#2196F3', alpha=0.7, edgecolor='white')
axes[0].set_title('Home Goals Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Goals')
axes[0].set_ylabel('Frequency')
axes[0].axvline(intl_matches['Home Goals'].mean(), color='red', 
                linestyle='--', label=f'Mean: {intl_matches["Home Goals"].mean():.2f}')
axes[0].legend()

# Away Goals distribution
axes[1].hist(intl_matches['Away Goals'].dropna(), bins=range(0, 15), 
             color='#FF5722', alpha=0.7, edgecolor='white')
axes[1].set_title('Away Goals Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Goals')
axes[1].axvline(intl_matches['Away Goals'].mean(), color='red', 
                linestyle='--', label=f'Mean: {intl_matches["Away Goals"].mean():.2f}')
axes[1].legend()

# Total goals per match
total_goals = intl_matches['Home Goals'] + intl_matches['Away Goals']
axes[2].hist(total_goals.dropna(), bins=range(0, 20), 
             color='#4CAF50', alpha=0.7, edgecolor='white')
axes[2].set_title('Total Goals Per Match', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Goals')
axes[2].axvline(total_goals.mean(), color='red', 
                linestyle='--', label=f'Mean: {total_goals.mean():.2f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_goal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Home Advantage Analysis

In [ ]:
# Hitung hasil pertandingan
valid = intl_matches.dropna(subset=['Home Goals', 'Away Goals']).copy()
valid['Result'] = 'Draw'
valid.loc[valid['Home Goals'] > valid['Away Goals'], 'Result'] = 'Home Win'
valid.loc[valid['Home Goals'] < valid['Away Goals'], 'Result'] = 'Away Win'

# Overall
result_counts = valid['Result'].value_counts()
result_pct = valid['Result'].value_counts(normalize=True) * 100

print('Overall Match Results:')
for result in ['Home Win', 'Draw', 'Away Win']:
    print(f'  {result}: {result_counts[result]:,} ({result_pct[result]:.1f}%)')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#2196F3', '#FFC107', '#FF5722']
axes[0].pie(result_pct[['Home Win', 'Draw', 'Away Win']], 
            labels=['Home Win', 'Draw', 'Away Win'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Overall Match Results', fontsize=14, fontweight='bold')

# Home advantage over time
valid['Year'] = valid['Date'].dt.year
valid['Decade'] = (valid['Year'] // 10) * 10
decade_home = valid.groupby('Decade')['Result'].apply(
    lambda x: (x == 'Home Win').mean() * 100
)
axes[1].plot(decade_home.index, decade_home.values, 'o-', 
             color='#2196F3', linewidth=2, markersize=8)
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Home Win Rate by Decade', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Home Win Rate (%)')
axes[1].set_ylim(30, 60)

plt.tight_layout()
plt.savefig('../outputs/figures/01_home_advantage.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. World Cup Winners History

In [ ]:
# World Cup winners
winners = world_cups['Winner'].dropna().value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
colors_wc = plt.cm.Set2(np.linspace(0, 1, len(winners)))
bars = ax.barh(winners.index[::-1], winners.values[::-1], color=colors_wc)
ax.set_xlabel('Number of Titles', fontsize=13)
ax.set_title('🏆 FIFA World Cup Winners', fontsize=16, fontweight='bold')

for bar, val in zip(bars, winners.values[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/01_wc_winners.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Goals per World Cup Trend

In [ ]:
wc_valid = world_cups[world_cups['Goals Scored'].notna()].copy()
wc_valid['Goals per Match'] = wc_valid['Goals Scored'] / wc_valid['Matches Played']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Total goals
axes[0].bar(wc_valid['Year'].astype(str), wc_valid['Goals Scored'], 
            color='#4CAF50', alpha=0.8, edgecolor='white')
axes[0].set_title('Total Goals per World Cup', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Total Goals')

# Goals per match
axes[1].plot(wc_valid['Year'], wc_valid['Goals per Match'], 'o-', 
             color='#FF5722', linewidth=2, markersize=6)
axes[1].set_title('Goals per Match Trend', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Goals per Match')
axes[1].axhline(y=wc_valid['Goals per Match'].mean(), color='gray', 
                linestyle='--', alpha=0.5, 
                label=f'Average: {wc_valid["Goals per Match"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_wc_goals_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Top Teams — All-Time Performance

In [ ]:
# Hitung performa per tim di pertandingan internasional
def calculate_team_stats(df, team):
    home = df[df['Home Team'] == team]
    away = df[df['Away Team'] == team]
    
    home_wins = (home['Home Goals'] > home['Away Goals']).sum()
    away_wins = (away['Away Goals'] > away['Home Goals']).sum()
    home_draws = (home['Home Goals'] == home['Away Goals']).sum()
    away_draws = (away['Away Goals'] == away['Home Goals']).sum()
    
    total_matches = len(home) + len(away)
    total_wins = home_wins + away_wins
    total_draws = home_draws + away_draws
    total_goals = home['Home Goals'].sum() + away['Away Goals'].sum()
    total_conceded = home['Away Goals'].sum() + away['Home Goals'].sum()
    
    return {
        'Team': team,
        'Matches': total_matches,
        'Wins': total_wins,
        'Draws': total_draws,
        'Losses': total_matches - total_wins - total_draws,
        'Win Rate': total_wins / total_matches if total_matches > 0 else 0,
        'Goals Scored': total_goals,
        'Goals Conceded': total_conceded,
        'Goal Diff': total_goals - total_conceded,
    }

# Get all unique teams
all_teams = set(valid['Home Team'].unique()) | set(valid['Away Team'].unique())
team_stats = pd.DataFrame([calculate_team_stats(valid, t) for t in all_teams])
team_stats = team_stats[team_stats['Matches'] >= 50]  # Min 50 matches
team_stats = team_stats.sort_values('Win Rate', ascending=False)

# Top 20 teams by win rate
top20 = team_stats.head(20)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(top20['Team'].iloc[::-1], top20['Win Rate'].iloc[::-1] * 100,
               color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)))
ax.set_xlabel('Win Rate (%)', fontsize=13)
ax.set_title('Top 20 Teams by Win Rate (min 50 matches)', fontsize=16, fontweight='bold')

for bar, val in zip(bars, top20['Win Rate'].iloc[::-1] * 100):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/01_top_teams.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. WC 2026 Participants Overview

In [ ]:
# WC 2026 groups visualization
print('FIFA World Cup 2026 — Group Draw')
print('=' * 60)
for group_name, group_df in wc2026_groups.groupby('Group'):
    teams = group_df[['Team', 'FIFA Ranking']].values
    team_str = ' | '.join([f'{t} (#{r})' for t, r in teams])
    print(f'  Group {group_name}: {team_str}')

print(f'\nTotal teams: {len(wc2026_groups)}')
print(f'Total groups: {wc2026_groups["Group"].nunique()}')

In [ ]:
# FIFA Ranking distribution of WC 2026 participants
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ranking distribution
axes[0].hist(wc2026_groups['FIFA Ranking'], bins=20, 
             color='#673AB7', alpha=0.7, edgecolor='white')
axes[0].set_title('FIFA Ranking Distribution — WC 2026 Teams', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('FIFA Ranking')
axes[0].set_ylabel('Count')
axes[0].axvline(wc2026_groups['FIFA Ranking'].median(), color='red', 
                linestyle='--', label=f'Median: {wc2026_groups["FIFA Ranking"].median():.0f}')
axes[0].legend()

# Group strength (avg ranking per group, lower = stronger)
group_strength = wc2026_groups.groupby('Group')['FIFA Ranking'].mean().sort_values()
colors_g = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(group_strength)))
axes[1].barh(group_strength.index[::-1], group_strength.values[::-1], color=colors_g)
axes[1].set_xlabel('Average FIFA Ranking (lower = stronger)', fontsize=12)
axes[1].set_title('Group Strength (Avg FIFA Ranking)', fontsize=14, fontweight='bold')

for i, (idx, val) in enumerate(group_strength[::-1].items()):
    axes[1].text(val + 0.5, i, f'{val:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/01_wc2026_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. WC 2026 Teams — Historical Performance

In [ ]:
# Performance of WC 2026 teams in recent international matches (2018-2022)
recent = valid[valid['Year'] >= 2018].copy()
wc2026_teams = wc2026_groups['Team'].tolist()

# Need to normalize team names for matching
import sys
sys.path.insert(0, '..')
from src.features import normalize_team_name, TEAM_NAME_MAP

# Normalize WC2026 team names to match international_matches
wc2026_teams_normalized = [normalize_team_name(t) for t in wc2026_teams]

recent_stats = []
for team_orig, team_norm in zip(wc2026_teams, wc2026_teams_normalized):
    stats = calculate_team_stats(recent, team_norm)
    stats['Team_Original'] = team_orig
    # Get FIFA Ranking
    rank_row = wc2026_groups[wc2026_groups['Team'] == team_orig]
    stats['FIFA_Ranking'] = rank_row.iloc[0]['FIFA Ranking'] if len(rank_row) > 0 else 99
    recent_stats.append(stats)

recent_df = pd.DataFrame(recent_stats)
recent_df = recent_df[recent_df['Matches'] > 0]
recent_df = recent_df.sort_values('Win Rate', ascending=False)

# Display top 20
print('WC 2026 Teams — Recent Form (2018-2022):')
recent_df[['Team_Original', 'FIFA_Ranking', 'Matches', 'Wins', 'Draws', 
           'Losses', 'Win Rate', 'Goal Diff']].head(20).to_string(index=False)

In [ ]:
# Scatter: Win Rate vs FIFA Ranking
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(recent_df['FIFA_Ranking'], recent_df['Win Rate'] * 100,
                     s=recent_df['Matches'] * 3, alpha=0.6,
                     c=recent_df['Goal Diff'], cmap='RdYlGn', 
                     edgecolors='gray', linewidth=0.5)

# Label top teams
for _, row in recent_df.head(15).iterrows():
    ax.annotate(row['Team_Original'], 
                (row['FIFA_Ranking'], row['Win Rate'] * 100),
                fontsize=9, ha='center', va='bottom',
                textcoords='offset points', xytext=(0, 8))

plt.colorbar(scatter, label='Goal Difference')
ax.set_xlabel('FIFA Ranking (lower = better)', fontsize=13)
ax.set_ylabel('Win Rate (%)', fontsize=13)
ax.set_title('WC 2026 Teams: FIFA Ranking vs Win Rate (2018-2022)\nBubble size = # matches',
             fontsize=14, fontweight='bold')
ax.invert_xaxis()  # Lower ranking on right

plt.tight_layout()
plt.savefig('../outputs/figures/01_wc2026_teams_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Insights Summary

Ringkasan temuan dari EDA ini:

1. **Home Advantage**: Tim tuan rumah menang ~45% pertandingan — signifikan. USA/Canada/Mexico punya keuntungan di WC 2026.
2. **Goals Trend**: Rata-rata gol per pertandingan menurun dari era awal (~4.5) ke era modern (~2.5).
3. **Top Teams**: Brazil, Germany, Argentina, France, dan Spain secara konsisten punya win rate tertinggi.
4. **WC Experience**: Tim dengan banyak pengalaman WC cenderung lebih baik.
5. **FIFA Ranking vs Performance**: Ada korelasi kuat antara FIFA Ranking dan win rate terkini.

**Next Step**: Notebook `02_feature_engineering.ipynb` — membangun fitur dari insight ini.